# Tirol Ski Resort Snow-Line Exposure Analysis

Same analytical shape as the NYC hydrant density project: load, explore, spatial join, aggregate, export.

**Data sources:**
- Official ski area boundaries — Land Tirol "URP Schigebietsgrenzen" (103 legally-defined areas)
- Tirol province outline — OpenStreetMap admin boundary (map background context only)
- Pistes — OpenStreetMap via Overpass
- On-mountain venues (restaurants/bars) — Overture Places, queried via DuckDB on S3
- Elevation — Land Tirol digital terrain model, via WCS (tiris)

**Chapters:**
1. Setup and data fetch
2. Load boundaries + pistes, rank resorts by size
3. Map all resort boundaries
4. Snow-line exposure — % below 1800m, km below 1800m, km above 1800m by difficulty, and summary counts
5. On-mountain venue counts
6. Venue density (venues per km²)
7. Visual density map
8. Export to GeoParquet

## 1. Setup and Data Fetch

In [ ]:
import os
for _var in ["LD_LIBRARY_PATH", "GDAL_DRIVER_PATH", "GDAL_DATA", "PROJ_LIB", "GDAL_PLUGINS_PATH"]:
    os.environ.pop(_var, None)

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
import rasterio
from rasterio.io import MemoryFile

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 15)

AUSTRIA_CRS = 31287
SNOW_RELIABILITY_THRESHOLD_M = 1800
ON_MOUNTAIN_DISTANCE_M = 100
MIN_AREA_KM2 = 1

print(f"GeoPandas version: {gpd.__version__}")

### 1a. Fetch Data

Runs `00_fetch_data.py`. Safe to run every time — it skips anything already downloaded.

In [ ]:
!python 00_fetch_data.py

## 2. Load Boundaries and Pistes, Rank by Size

In [ ]:
resorts = gpd.read_file("urp_schigebietsgrenzen.geojson")
pistes = gpd.read_file("data_pistes.gpkg")
print(f"Resorts: {resorts.shape}, CRS: {resorts.crs}")
print(f"Pistes:  {pistes.shape}, CRS: {pistes.crs}")

In [ ]:
rows = []
for _, row in resorts.iterrows():
    clipped = gpd.clip(pistes, gpd.GeoDataFrame(geometry=[row.geometry], crs=4326))
    km = clipped.to_crs(AUSTRIA_CRS).geometry.length.sum() / 1000 if len(clipped) else 0.0
    rows.append({"resort": row["NAME"], "piste_km": round(km, 1)})

ranking = pd.DataFrame(rows).sort_values("piste_km", ascending=False).reset_index(drop=True)
ranking.index += 1
print(f"Top 10 of {len(ranking)} resorts, ranked biggest to smallest by piste km")
print(f"(full ranking saved to 02_resort_ranking.csv):\n")
ranking.to_csv("02_resort_ranking.csv")
ranking.head(10)

## 3. Map All Resort Boundaries

The Tirol province outline is plotted first as background context.

In [ ]:
tirol_boundary = gpd.read_file("data_tirol_boundary.gpkg")

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
tirol_boundary.plot(ax=ax, color='#f2f2ee', edgecolor='#999999', linewidth=0.8)
resorts.plot(ax=ax, color='#cde6f5', edgecolor='#2b6ca3', linewidth=0.4)
ax.set_title(f'{len(resorts)} Official Ski Area Boundaries — Tirol', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('images/resort_boundaries_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Snow-Line Exposure

For each of Tirol's 103 official ski areas, this step samples ground elevation every 50 meters along
every mapped piste, then calculates what share of each resort's total piste length lies below the
1800m snow-reliability threshold — broken down by run difficulty. It reads vector geometry (piste
lines, via GDAL through GeoPandas) and raster elevation data (via rasterio) together, then collapses
the spatial detail into a plain tabular summary per resort. Resorts with no mapped piste data inside
their official boundary are flagged and excluded from the results that follow.

In [ ]:
def points_along_line(line, spacing_m=50):
    length_deg = line.length
    if length_deg == 0:
        return [line.centroid]
    n_points = max(2, int(length_deg * 111000 / spacing_m))
    return [line.interpolate(i / (n_points - 1), normalized=True) for i in range(n_points)]

def sample_elevations(dem_src, points):
    coords = [(p.x, p.y) for p in points]
    return np.array([v[0] for v in dem_src.sample(coords)])

with open("data_dem.tif", "rb") as f:
    dem_bytes = f.read()
print(f"DEM loaded: {len(dem_bytes)/1e6:.1f} MB")

In [ ]:
with MemoryFile(dem_bytes) as memfile, memfile.open() as dem_src:
    exposure_rows = []
    for _, row in resorts.iterrows():
        resort_poly = gpd.GeoDataFrame(geometry=[row.geometry], crs=4326)
        resort_pistes = gpd.clip(pistes, resort_poly)
        if len(resort_pistes) == 0:
            print(f"  {row['NAME']}: no data")
            continue
        per_difficulty = {}
        for line, difficulty in zip(resort_pistes.geometry, resort_pistes["difficulty"]):
            pts = points_along_line(line)
            elevations = sample_elevations(dem_src, pts)
            valid = elevations[elevations > -1000]
            if len(valid) == 0:
                continue
            seg_km = line.length * 111.0 * len(valid) / len(pts)
            below_frac = (valid < SNOW_RELIABILITY_THRESHOLD_M).mean()
            d = per_difficulty.setdefault(difficulty, {"total_km": 0.0, "below_km": 0.0})
            d["total_km"] += seg_km
            d["below_km"] += seg_km * below_frac
        total_km = sum(d["total_km"] for d in per_difficulty.values())
        below_km = sum(d["below_km"] for d in per_difficulty.values())
        above_km = total_km - below_km
        entry = {
            "resort": row["NAME"],
            "total_km": round(total_km, 1),
            "below_1800m_km": round(below_km, 1),
            "above_1800m_km": round(above_km, 1),
            "pct_below_1800m": round(100 * below_km / total_km, 1) if total_km > 0 else 0,
        }
        for diff, d in per_difficulty.items():
            entry[f"{diff}_above_km"] = round(d["total_km"] - d["below_km"], 1)
        exposure_rows.append(entry)

exposure = pd.DataFrame(exposure_rows)
print(f"\nProcessed {len(exposure)} of {len(resorts)} resorts with valid elevation data "
      f"({len(resorts) - len(exposure)} excluded -- no piste data mapped in OSM for those areas).")

### 4a. Ranked by % below 1800m (least exposed → most exposed)

In [ ]:
ranked_pct = exposure[["resort", "pct_below_1800m", "total_km"]].sort_values(
    "pct_below_1800m", ascending=True
).reset_index(drop=True)
ranked_pct.index += 1
print(f"Top 10 of {len(ranked_pct)} resorts, LEAST to MOST exposed (% below 1800m)")
print(f"(full ranking saved to 04a_pct_below_1800m.csv):\n")
ranked_pct.to_csv("04a_pct_below_1800m.csv")
ranked_pct.head(10)

### 4b. Ranked by km below 1800m (least → most)

In [ ]:
ranked_km = exposure[["resort", "below_1800m_km", "total_km"]].sort_values(
    "below_1800m_km", ascending=True
).reset_index(drop=True)
ranked_km.index += 1
print(f"Top 10 of {len(ranked_km)} resorts, LEAST to MOST km below 1800m")
print(f"(full ranking saved to 04b_km_below_1800m.csv):\n")
ranked_km.to_csv("04b_km_below_1800m.csv")
ranked_km.head(10)

### 4c. Terrain ABOVE 1800m by difficulty (most → least km above threshold)

The "safe" terrain, broken down by run difficulty.

In [ ]:
above_cols = [c for c in exposure.columns if c.endswith("_above_km") and c != "above_1800m_km"]
difficulty_above = exposure[["resort", "above_1800m_km"] + above_cols].sort_values(
    "above_1800m_km", ascending=False
).reset_index(drop=True)
difficulty_above.index += 1
print(f"Top 10 of {len(difficulty_above)} resorts, MOST to LEAST km above 1800m by difficulty")
print(f"(full ranking saved to 04c_difficulty_above_1800m.csv):\n")
difficulty_above.to_csv("04c_difficulty_above_1800m.csv")
difficulty_above.head(10)

### 4d. Summary Counts

A compact view of the same underlying exposure data: how many resorts fall into each risk
category, rather than reading through the individual rankings above.

In [ ]:
summary_counts = pd.DataFrame([
    {"metric": "Resorts 100% below 1800m (fully exposed)",
     "count": int((exposure["pct_below_1800m"] == 100).sum())},
    {"metric": "Resorts ≥50% below 1800m (majority exposed)",
     "count": int((exposure["pct_below_1800m"] >= 50).sum())},
    {"metric": "Resorts with ≥50 km of piste above 1800m",
     "count": int((exposure["above_1800m_km"] >= 50).sum())},
])
summary_counts["pct_of_total"] = (100 * summary_counts["count"] / len(exposure)).round(1)
print(f"Summary counts across {len(exposure)} resorts with valid elevation data:\n")
summary_counts.to_csv("04d_summary_counts.csv", index=False)
summary_counts

## 5. On-Mountain Venue Counts

Restaurants and bars within 100m of an actual piste — not a flat count within the whole
resort boundary, which would mostly measure the base village rather than the mountain itself.

In [ ]:
venues = gpd.read_file("data_venues.gpkg")
print(f"Venues loaded: {venues.shape}")

venue_rows = []
for _, row in resorts.iterrows():
    resort_poly = gpd.GeoDataFrame(geometry=[row.geometry], crs=4326)
    resort_pistes = gpd.clip(pistes, resort_poly)
    if len(resort_pistes) == 0 or len(venues) == 0:
        venue_rows.append({"resort": row["NAME"], "on_mountain_venue_count": 0})
        continue
    pistes_m = resort_pistes.to_crs(AUSTRIA_CRS)
    venues_m = venues.to_crs(AUSTRIA_CRS)
    piste_buffer = pistes_m.geometry.buffer(ON_MOUNTAIN_DISTANCE_M).union_all()
    count = int(venues_m.geometry.within(piste_buffer).sum())
    venue_rows.append({"resort": row["NAME"], "on_mountain_venue_count": count})

venue_counts = pd.DataFrame(venue_rows).sort_values(
    "on_mountain_venue_count", ascending=False
).reset_index(drop=True)
venue_counts.index += 1
print(f"Top 10 of {len(venue_counts)} resorts, MOST to LEAST on-mountain venues")
print(f"(full ranking saved to 05_venue_counts.csv):\n")
venue_counts.to_csv("05_venue_counts.csv")
venue_counts.head(10)

## 6. Venue Density (venues per km²)

Same normalization principle as the hydrant density project: raw counts favor bigger resorts.
Dividing by area makes different-sized resorts fairly comparable — but for very small polygons,
this ratio becomes unstable. One resort in the official dataset has an area of ~0.00002 km²
(a data artifact), which produced a nonsensical density in the tens of thousands per km².
Resorts under 1 km² are excluded from this ranking for that reason.

In [ ]:
resorts_m = resorts.to_crs(AUSTRIA_CRS)
resorts["area_km2"] = resorts_m.geometry.area / 1e6

density = resorts[["NAME", "area_km2"]].rename(columns={"NAME": "resort"}).merge(
    venue_counts, on="resort", how="left"
)
density["venue_density_per_km2"] = (
    density["on_mountain_venue_count"] / density["area_km2"]
).round(2)

density_filtered = density[density["area_km2"] >= MIN_AREA_KM2]
excluded_count = len(density) - len(density_filtered)

density_ranked = density_filtered.sort_values("venue_density_per_km2", ascending=False).reset_index(drop=True)
density_ranked.index += 1
print(f"Top 10 of {len(density_ranked)} resorts ≥{MIN_AREA_KM2} km², MOST to LEAST venue density per km²")
print(f"({excluded_count} smaller resorts excluded -- density is unstable on tiny areas)")
print(f"(full ranking saved to 06_venue_density.csv):\n")
density_ranked.to_csv("06_venue_density.csv")
density_ranked.head(10)

## 7. Visual Density Map

Same 1 km² minimum-area filter as Chapter 6. The color scale is fixed to defined steps
(0–3, in increments of 0.5) rather than scaling automatically to the data's min/max — this
keeps the map readable and stable regardless of any single outlier value.

In [ ]:
resorts_with_density = resorts.merge(
    density[["resort", "venue_density_per_km2"]], left_on="NAME", right_on="resort", how="left"
)
resorts_for_density = resorts_with_density[resorts_with_density["area_km2"] >= MIN_AREA_KM2]
excluded = len(resorts_with_density) - len(resorts_for_density)

bounds = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
cmap = plt.cm.YlOrRd
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
tirol_boundary.plot(ax=ax, color='#f2f2ee', edgecolor='#999999', linewidth=0.8)
resorts_for_density.plot(
    ax=ax, column='venue_density_per_km2', cmap=cmap, norm=norm, legend=True,
    edgecolor='white', linewidth=0.3,
    legend_kwds={'ticks': bounds, 'label': 'Venues per km²'},
    missing_kwds={"color": "#eeeeee"},
)
ax.set_title(f'On-Mountain Venue Density (per km²) by Resort\n'
             f'({excluded} resorts under {MIN_AREA_KM2} km² excluded)', fontsize=12)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('images/venue_density_choropleth.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Export to GeoParquet

In [ ]:
final = resorts_with_density.merge(exposure, left_on="NAME", right_on="resort", how="left")
final = final.drop(columns=[c for c in ["resort_x", "resort_y", "resort"] if c in final.columns])

output_path = "tirol_snowline_results.parquet"
final.to_parquet(output_path)

result = gpd.read_parquet(output_path)
print(f"Exported to: {output_path}")
print(f"Shape: {result.shape}")
print(f"Columns: {list(result.columns)}")

## Summary

In this notebook you:
1. Fetched all data sources (self-contained, skips already-cached files)
2. Loaded official Tirol ski area boundaries and OSM piste data, ranked all 103 by piste km
3. Mapped all resort boundaries with the Tirol province outline as geographic context
4. Sampled elevation along every piste (via GDAL/GeoPandas + rasterio) to compute snow-line
   exposure — by %, by km, by safe terrain per difficulty level, and as summary risk-category
   counts; 94 of 103 resorts had usable piste data, 9 were excluded for having none mapped in OSM
5. Counted on-mountain venues within 100m of pistes (Overture Places via DuckDB/S3)
6. Computed area-normalized venue density, excluding resorts under 1 km² (unstable ratio
   on near-zero-area polygons)
7. Visualized density as a choropleth with a fixed, stepped color scale
8. Exported the combined result to GeoParquet

## What I learned

The biggest lesson was architectural: my first version made a fresh Overpass/DEM/Overture
query for every single resort, which meant 100+ network round-trips and painfully slow runs.
Rewriting it to fetch each data source once for the whole province, then clip and sample
locally per resort, cut the runtime from over 30 minutes to under 90 seconds — the same
"push work to the source, not to Python" lesson from the hydrant project's DuckDB
aggregation, just applied at pipeline scale. I also learned to distrust convenient
shortcuts: a radius around each resort's center point badly over- or under-shot real piste
extent in Tirol's densely interconnected terrain, and only switching to the official
government boundary polygons fixed it properly. Finally, a density calculation is only as
good as its denominator — one resort with a near-zero official area produced a nonsensical
density value that would have skewed the whole map if I hadn't filtered it out and
documented why.